
# Descarga histórica — Futuros Agropecuarios (CEM2 / Matba Rofex)

Fuente: **CEM2**, la API pública y gratuita de Matba Rofex que unifica el
histórico de precios de Rofex (ex-CEM) y Matba (ex-DataCenter) desde 2024,
con datos históricos cargados desde 1909.

- Visor: https://cem.matbarofex.com.ar/
- API:   https://cem-api-prd.azurewebsites.net (sin autenticación)

Endpoint usado: `GET /api/v1/closing-prices` — devuelve velas diarias
completas (apertura, máximo, mínimo, cierre, ajuste, volumen, interés
abierto) para un rango de fechas, paginado.

## Cómo filtramos "Agropecuario - Futuros"
El parámetro `product` de la API espera el id exacto tal como aparece en
`/api/v1/products` (con espacios, ej. `"SOJ Dolar MATba"`), y en las
pruebas no pareció filtrar de forma confiable. Para no depender de eso,
este notebook:

1. Trae la lista de productos (`/api/v1/products`) y arma el conjunto de
   productos con `segment == "Agropecuario"`.
2. Trae los cierres diarios del rango completo, paginando.
3. Filtra del lado cliente:
   - `product` esté en el conjunto de productos agropecuarios, Y
   - el `symbol` tenga la forma clásica de un futuro Matba-Rofex
     (`PRODUCTO.MERCADO/MESAÑO`, ej. `SOJ.ROS/MAY24`), para excluir los
     productos "Disponible" (mercado spot/físico, no futuro).


In [1]:

import json
import re
import time
from datetime import date, timedelta
from pathlib import Path

import pandas as pd
import requests


## Configuración

In [2]:

BASE_URL = "https://cem-api-prd.azurewebsites.net"

FECHA_DESDE_GLOBAL = date(2024, 1, 1)
FECHA_HASTA_GLOBAL = date.today()

# Partimos el pedido en ventanas mensuales para no mandar rangos gigantes
# de una sola vez.
CHUNK_DAYS = 30

PAGE_SIZE = 1000  # tamaño de pagina para /closing-prices

SLEEP_SECONDS = 0.3
TIMEOUT = 30

# Patron de symbol de un futuro clasico Matba-Rofex: PRODUCTO.MERCADO/MESANIO
# (ej. SOJ.ROS/MAY24, CRN.CME/NOV26, TRI.ROS/DIC26)
PATRON_FUTURO_AGRO = re.compile(r"^[A-Z]+\.[A-Z]+/[A-Z]{3}\d{2}$")

OUTPUT_DIR = Path("cem2_futuros_agro")
RAW_DIR = OUTPUT_DIR / "raw_chunks"
FINAL_CSV = OUTPUT_DIR / "futuros_agropecuario_2024_a_hoy.csv"

RAW_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {"Accept": "application/json"}


## Paso 1 — Productos agropecuarios (referencia)

In [3]:

resp = requests.get(f"{BASE_URL}/api/v1/products", headers=HEADERS, timeout=TIMEOUT)
resp.raise_for_status()
productos = resp.json()["data"]

productos_agro = {p["id"] for p in productos if p.get("segment") == "Agropecuario"}

print(f"Total de productos: {len(productos)}")
print(f"Productos con segmento Agropecuario: {len(productos_agro)}")
for p in sorted(productos_agro):
    print(f"  - {p}")


Total de productos: 36
Productos con segmento Agropecuario: 19
  - CRN Matba
  - GIR
  - GIR ARS
  - MAI Disponible
  - MAI Dolar MATba
  - MAI MIN
  - MAI Pesos MATba
  - SOJ Disponible
  - SOJ Dolar MATba
  - SOJ MIN
  - SOJ Pesos MATba
  - SOR Disponible
  - SORGO
  - SORGO ARS
  - SOY Matba
  - TRI Disponible
  - TRI Dolar MATba
  - TRI MIN
  - TRI Pesos MATba


## Paso 2 — Funciones de descarga paginada por ventana de fechas

In [4]:

def generar_ventanas(desde: date, hasta: date, chunk_days: int):
    ventanas = []
    cursor = desde
    while cursor <= hasta:
        fin_ventana = min(cursor + timedelta(days=chunk_days - 1), hasta)
        ventanas.append((cursor, fin_ventana))
        cursor = fin_ventana + timedelta(days=1)
    return ventanas


def nombre_chunk(fecha_desde: date, fecha_hasta: date) -> Path:
    return RAW_DIR / f"{fecha_desde.isoformat()}_{fecha_hasta.isoformat()}.json"


def fetch_ventana(session: requests.Session, fecha_desde: date, fecha_hasta: date):
    # Trae TODAS las paginas de closing-prices para una ventana de fechas.
    registros = []
    page = 1
    while True:
        params = {
            "from": fecha_desde.isoformat(),
            "to": fecha_hasta.isoformat(),
            "page": page,
            "pageSize": PAGE_SIZE,
        }
        resp = session.get(f"{BASE_URL}/api/v1/closing-prices",
                            headers=HEADERS, params=params, timeout=TIMEOUT)
        resp.raise_for_status()
        data = resp.json().get("data", [])
        registros.extend(data)

        if len(data) < PAGE_SIZE:
            break  # ultima pagina (trajo menos de lo pedido)
        page += 1
        time.sleep(SLEEP_SECONDS)

    return registros



## Paso 3 — Descarga con resumen

Cada ventana mensual se guarda como su propio `.json` en `raw_chunks/`. Si
el notebook se corta, al volver a correr esta celda salta lo ya
descargado.


In [5]:

ventanas = generar_ventanas(FECHA_DESDE_GLOBAL, FECHA_HASTA_GLOBAL, CHUNK_DAYS)
print(f"Total de ventanas a cubrir: {len(ventanas)} "
      f"({FECHA_DESDE_GLOBAL} -> {FECHA_HASTA_GLOBAL}, bloques de {CHUNK_DAYS} dias)")

session = requests.Session()
errores = []

for i, (v_desde, v_hasta) in enumerate(ventanas, start=1):
    destino = nombre_chunk(v_desde, v_hasta)

    if destino.exists():
        print(f"[{i}/{len(ventanas)}] {v_desde} -> {v_hasta}: ya descargado, se omite.")
        continue

    print(f"[{i}/{len(ventanas)}] {v_desde} -> {v_hasta}: descargando...")
    try:
        registros = fetch_ventana(session, v_desde, v_hasta)
        destino.write_text(json.dumps(registros, ensure_ascii=False), encoding="utf-8")
        print(f"    -> {len(registros)} registros totales (todos los productos)")
    except requests.RequestException as e:
        print(f"    -> ERROR, se sigue con la proxima ventana: {e}")
        errores.append((v_desde, v_hasta, str(e)))

print()
print("Listo.")
if errores:
    print(f"{len(errores)} ventana(s) fallaron:")
    for v_desde, v_hasta, err in errores:
        print(f"  - {v_desde} -> {v_hasta}: {err}")
    print("Volve a correr la celda para reintentarlas (las exitosas se saltean).")


Total de ventanas a cubrir: 34 (2024-01-01 -> 2026-09-22, bloques de 30 dias)
[1/34] 2024-01-01 -> 2024-01-30: descargando...
    -> 5281 registros totales (todos los productos)
[2/34] 2024-01-31 -> 2024-02-29: descargando...
    -> 5861 registros totales (todos los productos)
[3/34] 2024-03-01 -> 2024-03-30: descargando...
    -> 6210 registros totales (todos los productos)
[4/34] 2024-03-31 -> 2024-04-29: descargando...
    -> 6505 registros totales (todos los productos)
[5/34] 2024-04-30 -> 2024-05-29: descargando...
    -> 6656 registros totales (todos los productos)
[6/34] 2024-05-30 -> 2024-06-28: descargando...
    -> 6256 registros totales (todos los productos)
[7/34] 2024-06-29 -> 2024-07-28: descargando...
    -> 5072 registros totales (todos los productos)
[8/34] 2024-07-29 -> 2024-08-27: descargando...
    -> 6509 registros totales (todos los productos)
[9/34] 2024-08-28 -> 2024-09-26: descargando...
    -> 6789 registros totales (todos los productos)
[10/34] 2024-09-27 -> 

## Paso 4 — Consolidar, filtrar solo futuros agropecuarios, y guardar

In [6]:

todos_los_registros = []
for archivo in sorted(RAW_DIR.glob("*.json")):
    todos_los_registros.extend(json.loads(archivo.read_text(encoding="utf-8")))

df_todo = pd.DataFrame(todos_los_registros)
print(f"Filas totales descargadas (todos los productos, agro + financieros): {len(df_todo)}")

# Filtro: producto con segmento Agropecuario Y symbol con forma de futuro
# (excluye "Disponible" = spot/fisico)
es_agro = df_todo["product"].isin(productos_agro)
es_futuro = df_todo["symbol"].str.match(PATRON_FUTURO_AGRO)

df = df_todo[es_agro & es_futuro].copy()
df["dateTime"] = pd.to_datetime(df["dateTime"])
df = df.drop_duplicates(subset=["dateTime", "symbol"]).sort_values(["dateTime", "symbol"])
df = df.reset_index(drop=True)

print(f"Filas de futuros agropecuarios luego del filtro: {len(df)}")
print(f"Rango de fechas: {df['dateTime'].min()} -> {df['dateTime'].max()}" if not df.empty else "Sin datos.")
print(f"Productos incluidos: {sorted(df['product'].unique())}" if not df.empty else "")
df.head()


Filas totales descargadas (todos los productos, agro + financieros): 187432
Filas de futuros agropecuarios luego del filtro: 24383
Rango de fechas: 2024-01-02 00:00:00+00:00 -> 2026-09-21 00:00:00+00:00
Productos incluidos: ['CRN Matba', 'MAI Disponible', 'MAI Dolar MATba', 'MAI MIN', 'SOJ Disponible', 'SOJ Dolar MATba', 'SOJ MIN', 'SOR Disponible', 'SORGO', 'SOY Matba', 'TRI Disponible', 'TRI Dolar MATba', 'TRI MIN']


,dateTime,symbol,settlement,high,low,open,close,openInterest,openInterestChange,change,...,tradeCount,impliedRate,previousClose,unitsOpenInterest,unitsOpenInterestChange,unitsVolume,optionType,strikePrice,underlying,product
0,2024-01-02 00:00:00+00:00,CRN.CME/ABR24,187.2,190.5,187.2,190.0,187.2,14430.0,-1738.0,-5.3,...,28,NaN,192.5,72150.0,-8690.0,9180.0,None,NaN,None,CRN Matba
1,2024-01-02 00:00:00+00:00,CRN.CME/FEB24,184.0,0.0,0.0,0.0,0.0,306.0,0.0,-2.0,...,0,NaN,186.0,1530.0,0.0,0.0,None,NaN,None,CRN Matba
2,2024-01-02 00:00:00+00:00,CRN.CME/JUN24,190.7,0.0,0.0,0.0,0.0,4729.0,0.0,-3.2,...,0,NaN,193.9,23645.0,0.0,0.0,None,NaN,None,CRN Matba
3,2024-01-02 00:00:00+00:00,MAI.MIN/ABR24,183.0,183.5,180.5,180.5,183.2,215.0,3.0,-0.7,...,4,NaN,183.7,2150.0,30.0,70.0,None,NaN,None,MAI MIN
4,2024-01-02 00:00:00+00:00,MAI.MIN/JUL24,174.0,174.1,174.1,174.1,174.1,98.0,2.0,-1.5,...,1,NaN,175.5,980.0,20.0,20.0,None,NaN,None,MAI MIN


## Chequeo rápido de huecos

In [7]:

if not df.empty:
    dias_con_datos = set(df["dateTime"].dt.date)
    dias_habiles_esperados = pd.bdate_range(FECHA_DESDE_GLOBAL, FECHA_HASTA_GLOBAL).date
    faltantes = sorted(set(dias_habiles_esperados) - dias_con_datos)

    print(f"Dias habiles en el rango: {len(dias_habiles_esperados)}")
    print(f"Dias habiles con al menos un registro: {len(dias_con_datos & set(dias_habiles_esperados))}")
    print(f"Dias habiles SIN ningun registro: {len(faltantes)}")
    if faltantes:
        print("(pueden ser feriados argentinos o dias sin operaciones en ningun contrato agro)")
        print("Primeros 20 faltantes:", faltantes[:20])


Dias habiles en el rango: 712
Dias habiles con al menos un registro: 661
Dias habiles SIN ningun registro: 51
(pueden ser feriados argentinos o dias sin operaciones en ningun contrato agro)
Primeros 20 faltantes: [datetime.date(2024, 1, 1), datetime.date(2024, 2, 12), datetime.date(2024, 2, 13), datetime.date(2024, 3, 28), datetime.date(2024, 3, 29), datetime.date(2024, 4, 1), datetime.date(2024, 4, 2), datetime.date(2024, 5, 1), datetime.date(2024, 6, 17), datetime.date(2024, 6, 20), datetime.date(2024, 6, 21), datetime.date(2024, 7, 9), datetime.date(2024, 10, 11), datetime.date(2024, 11, 18), datetime.date(2024, 12, 24), datetime.date(2024, 12, 25), datetime.date(2024, 12, 31), datetime.date(2025, 1, 1), datetime.date(2025, 3, 3), datetime.date(2025, 3, 4)]


## Guardar resultado final

In [8]:

if not df.empty:
    df.to_csv(FINAL_CSV, index=False, encoding="utf-8")
    print(f"Guardado: {FINAL_CSV.resolve()}  ({len(df)} filas)")
else:
    print("No hay datos para guardar todavia.")


Guardado: C:\Users\Lenovo\Downloads\cem2_futuros_agro\futuros_agropecuario_2024_a_hoy.csv  (24383 filas)
